In [237]:
from __future__ import annotations
from pathlib import Path
import re
import networkx as nx

#input_file = Path(".") / "example1_p1-5.txt"
#input_file = Path(".") / "example2_p2-2.txt"
input_file = Path(".") / "input.txt"

device_graph = nx.DiGraph()

def count_all_paths(source_node_name, target_node_name: str, g: nx.Graph) -> int:
    '''Count all possible paths going from source to target node
    It returns 0 if there is no matching route.
    Args:
        source_node_name (str): Name of the source node.
        target_node_name (str): Name of the target node.
    Return:
        int: Number of different paths
    '''
    try:
        return sum(1 for _ in nx.all_simple_paths(g, source_node_name, target_node_name))
    except nx.NodeNotFound:
        return 0

def count_all_paths_with_itermediate_nodes(source_node_name, target_node_name: str, intermediate_nodes: list, g: nx.Graph) -> int:
    '''Count all possible paths going from source to target node
    It returns 0 if there is no matching route.
    Calculation takes nodes in backwards from target towards the source node and sum up the path count.
    Args:
        source_node_name (str): Name of the source node.
        target_node_name (str): Name of the target node.
        intermediate_nodes (list): List of node names that each path must go through.
    Return:
        int: Number of different paths
    '''
    # Input validation
    for node in [source_node_name, target_node_name] + intermediate_nodes:
        if not g.has_node(node):
            raise ValueError(f"The node '{node}' does not exist in the graph.")
    
    g.nodes[target_node_name]["path_count"] = 1    

    # Sort node on each depth level by reversed topological order
    sorted_nodes = list(reversed(list(nx.topological_sort(g))))

    # Remove nodes preceeding the target node (including itself),
    # to make sure calculation is started from a node coming after the target node.
    for target_index in range(len(sorted_nodes)):
        if sorted_nodes[target_index] == target_node_name:
            break
    sorted_nodes = sorted_nodes[target_index + 1:]

    # Count the paths backdards (from target to source)
    for node in sorted_nodes:
        # Sum up possible paths from descendant nodes
        path_count = 0
        for descendant_node in g.successors(node):
            path_count += g.nodes[descendant_node]["path_count"]
        g.nodes[node]["path_count"] = path_count

        # Limit the counting to the intermediate nodes by
        # setting the path_count to zero for same depth level nodes.
        if node in intermediate_nodes:
            for n in sorted_nodes:
                if n != node:
                    g.nodes[n]["path_count"] = 0

    return g.nodes[source_node_name]["path_count"] if "path_count" in g.nodes[source_node_name] else 0

# Initialize and read input
with input_file.open(mode="r", encoding="utf-8") as file:
    for line in file:
        parent_node_name, connections = line.split(":")
        node_connections = connections.strip().split(" ")
        for child_node_name in node_connections:
            device_graph.add_edge(parent_node_name, child_node_name)

print(f"Part1 answer: {count_all_paths('you', 'out', device_graph)}")
print(f"Part2 answer: {count_all_paths_with_itermediate_nodes('svr', 'out', ['fft', 'dac'], device_graph)}")

Part1 answer: 690
Part2 answer: 557332758684000
